# Underwater Computer Vision With FathomNet (Master Notebook)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Spiffical/fathomnet-underwater-vision-tutorial/blob/main/notebooks/fathomnet_underwater_vision_tutorial_master.ipynb)

This notebook introduces computer vision workflows for underwater imagery using FathomNet-derived data. You will train and compare image classification, object detection, instance segmentation, and promptable segmentation workflows.

You only need a mathematical foundation to start. You can follow the guided path from classification to segmentation, or jump into the section that matches your comfort level.

You will work with images derived from the [FathomNet Database](https://database.fathomnet.org/fathomnet/#/), an expert-annotated underwater image database designed to support marine science and machine learning. The problem is simple to state and hard to solve: given an underwater image, decide what organisms or biological structures are present, where they are, and sometimes which pixels belong to each object.

Underwater imagery is not just "ImageNet, but blue." Vehicle lights, water-column attenuation, turbidity, motion blur, scale changes, partial animals, transparent bodies, and long-tailed taxonomy all make this a useful stress test for computer vision.

Main source anchors:

- FathomNet Database: https://database.fathomnet.org/fathomnet/#/
- FathomNet data use policy: https://www.fathomnet.org/datause
- Ultralytics training guide: https://docs.ultralytics.com/modes/train/
- Ultralytics classification task: https://docs.ultralytics.com/tasks/classify/
- Ultralytics detection format: https://docs.ultralytics.com/datasets/detect/
- Ultralytics segmentation format: https://docs.ultralytics.com/datasets/segment/
- SAM3 repository: https://github.com/facebookresearch/sam3
- SAM3 image predictor example: https://github.com/facebookresearch/sam3/blob/main/examples/sam3_image_predictor_example.ipynb

This instructor copy includes worked code solutions and short answer-key notes. Use the non-master notebook for the live participant version.

## How To Use This Notebook

Start with setup and the first image grid so the visual problem is concrete. After that, you can follow the notebook in order or jump to the section that matches your comfort level.

The main path is:

1. inspect FathomNet-derived underwater images and annotations;
2. warm up with a pretrained YOLO model on a familiar image;
3. train a small classifier on organism crops;
4. train and evaluate a binary object detector;
5. train and evaluate an instance segmentation model;
6. try text-prompt segmentation with cached or live SAM3 outputs.

The sections are intentionally independent. If you skip ahead, run the short **section bootstrap** cell at the top of that section first.

The default notebook behavior is:

- use a prebuilt compact data bundle,
- run live YOLO training only when a GPU is available,
- fall back to cached training curves and cached SAM3-style outputs when a GPU, model weights, or Hugging Face access are unavailable.

Vocabulary you will see:

- **YOLO** means "You Only Look Once"; in this notebook it refers to Ultralytics YOLO models for classification, detection, and segmentation.
- A **bounding box** is a rectangle around an object, usually represented by center point, width, and height.
- **Segmentation** means predicting pixels or regions, not just a class or rectangle.
- **Instance segmentation** means predicting a separate mask for each object instance.
- **COCO** means "Common Objects in Context"; here it mostly refers to a widely used JSON annotation format for images, categories, bounding boxes, and segmentations.
- **SAM3** is a promptable segmentation model in Meta's Segment Anything family; here you will use text prompts such as `"fish"` or `"small crab"`.

### Suggested Instructor Timing

For a three-hour live session, a reasonable pacing target is:

1. `0-15 min`: setup, runtime check, and first look at FathomNet imagery.
2. `15-30 min`: YOLO warm-up on a familiar image.
3. `30-50 min`: dataset and annotation exploration.
4. `50-75 min`: classification from organism crops.
5. `75-115 min`: binary object detection with YOLO.
6. `115-150 min`: instance segmentation with YOLO.
7. `150-175 min`: SAM3 text-prompt segmentation.
8. `175-180 min`: wrap-up and extension ideas.

## Setup And Runtime Check

First locate the repository, optionally install dependencies, load the small bundle, and inspect the GPU.

GitHub repository for this tutorial:

https://github.com/Spiffical/fathomnet-underwater-vision-tutorial

The notebook works in two common modes:

- **Colab from GitHub:** the setup cell clones the repository into `/content/fathomnet_underwater_vision_tutorial`.
- **Local Jupyter:** start Jupyter from the cloned repository, or from a child directory inside it.

Data storage model for the session:

- The canonical data bundle is `data/fathomnet_underwater_tutorial_bundle.zip` in the repository.
- For Colab, the default `BUNDLE_URL` points to the zip in this GitHub repository.
- In Colab, you download your own copy into the temporary runtime, then the notebook extracts it under `data/fathomnet_underwater_tutorial_bundle`.
- Nothing is downloaded live from FathomNet during the session.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/Spiffical/fathomnet-underwater-vision-tutorial"
GITHUB_BRANCH = "main"
PROJECT_DIR_NAME = "fathomnet_underwater_vision_tutorial"

def _candidate_roots():
    cwd = Path.cwd().resolve()
    yield cwd
    yield from cwd.parents
    yield Path("/content") / PROJECT_DIR_NAME

REPO_ROOT = None
for candidate in _candidate_roots():
    if (candidate / "scripts").exists() and (candidate / "notebooks").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None and "google.colab" in sys.modules:
    clone_dir = Path("/content") / PROJECT_DIR_NAME
    if not clone_dir.exists():
        subprocess.check_call([
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            GITHUB_BRANCH,
            f"{GITHUB_REPO_URL}.git",
            str(clone_dir),
        ])
    REPO_ROOT = clone_dir

if REPO_ROOT is None:
    raise RuntimeError("Could not find the tutorial repo root. Start Jupyter from the cloned repo.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")

### Dependencies And Runtime

Now check the Python dependencies, inspect the GPU/runtime, and set the random seed. In Colab this cell installs missing packages; locally it only checks what is already importable.

In [ ]:
from scripts.tutorial_setup import (
    detect_runtime,
    download_tutorial_bundle,
    ensure_dependencies,
    print_runtime_summary,
    set_reproducible_seed,
)

# Local runs use the repo-local zip automatically. In Colab, this URL is the
# fallback if the zip is not already present after cloning.
BUNDLE_URL = "https://github.com/Spiffical/fathomnet-underwater-vision-tutorial/raw/main/data/fathomnet_underwater_tutorial_bundle.zip"
LOCAL_BUNDLE_ZIP = REPO_ROOT / "data" / "fathomnet_underwater_tutorial_bundle.zip"

IN_COLAB = "google.colab" in sys.modules
INSTALL_DEPENDENCIES = IN_COLAB
RUN_LIVE_TRAINING = False  # updated after the runtime check

ensure_dependencies(install=INSTALL_DEPENDENCIES, extra_pip_args=("--quiet",))
RUNTIME = print_runtime_summary(detect_runtime())
RUN_LIVE_TRAINING = bool(RUNTIME.get("has_cuda", False))
set_reproducible_seed(42)

print(f"RUN_LIVE_TRAINING = {RUN_LIVE_TRAINING}")

### Download Or Unpack The Tutorial Bundle

This cell prepares the compact FathomNet-derived data bundle. In Colab, each participant gets a temporary runtime copy. Locally, the notebook uses the zip already stored in the repository when available.

In [ ]:
BUNDLE_ROOT = download_tutorial_bundle(
    bundle_url=BUNDLE_URL or None,
    bundle_zip_path=LOCAL_BUNDLE_ZIP if LOCAL_BUNDLE_ZIP.exists() else None,
    output_dir=REPO_ROOT / "data" / "fathomnet_underwater_tutorial_bundle",
)

print(f"Bundle root: {BUNDLE_ROOT}")
print((BUNDLE_ROOT / "manifest.json").read_text()[:1200])

### Train, Validation, And Test Splits

Most supervised machine-learning workflows keep separate data splits:

- **Training set:** examples used to update the model parameters by gradient descent.
- **Validation set:** examples used during development to choose hyperparameters, compare runs, tune thresholds, and select the best checkpoint.
- **Test set:** examples held back until the end for a final estimate of performance on data you did not optimize against.

In symbols, training chooses parameters

$$\hat{\theta}=\arg\min_\theta \frac{1}{n_{\mathrm{train}}}\sum_{i\in \mathrm{train}}\ell(f_\theta(x_i),y_i),$$

while validation estimates whether those parameters are useful away from the data that supplied the gradients. Ultralytics saves `weights/best.pt` as the checkpoint that performs best on the validation set during training. That is usually the checkpoint you evaluate or fine-tune from next, while `weights/last.pt` is simply the final epoch.

This compact tutorial bundle uses train/validation splits for live exercises. In a real project, keep a separate test split untouched until your modeling choices are fixed.

## First Look At FathomNet Imagery

Before training anything, look at the data. These are full underwater images from the compact FathomNet-derived bundle. Some organisms are obvious, some are tiny, and some are visually ambiguous even for a human.

As you scan the grid, ask:

- What would count as an "object"?
- Which organisms are easy to crop and classify?
- Which organisms need a bounding box?
- Which organisms would really need a pixel mask?

### Master Notes: First Look

- Let participants look before naming every task. Good early prompts are: "What is easy to see?", "What is small?", and "What would be hard to annotate consistently?"
- This section intentionally uses full images before crops, boxes, or masks so the modeling choices feel motivated by the visual problem.
- If a participant says the image has many unlabeled things, that is a useful opening for incomplete labels and distribution shift.

In [ ]:
from scripts.tutorial_data import get_task_paths
from scripts.tutorial_viz import show_image_grid

detect_paths = get_task_paths("detect", BUNDLE_ROOT)
segment_paths = get_task_paths("segment", BUNDLE_ROOT)
classification_paths = get_task_paths("classification", BUNDLE_ROOT)
coco_paths = get_task_paths("coco", BUNDLE_ROOT)

first_look_images = sorted((detect_paths["root"] / "images" / "val").glob("*.jpg"))[:8]
show_image_grid(
    first_look_images,
    titles=[path.stem[:8] for path in first_look_images],
    columns=4,
)

### Visible Helper: Training Arguments

You are encouraged to edit this function. It is deliberately small: the goal is to connect a few knobs to training behavior.

In [ ]:
def build_train_args(
    *,
    n_epochs=10,
    imgsz=320,
    batch=8,
    lr0=0.001,
    patience=3,
    workers=0,
    project="runs/tutorial",
    name="experiment",
    seed=42,
):
    '''Collect the main training hyperparameters in one visible place.

    Think of this as choosing an optimizer trajectory through parameter space:

        theta_{t+1} = theta_t - eta * grad L(theta_t)

    where `lr0` controls the initial step size eta, `batch` controls how noisy
    the gradient estimate is, and `n_epochs` controls how many passes you make
    through the finite training sample.

    Ultralytics expects this argument to be named `epochs`, so the returned
    dictionary maps the classroom-facing `n_epochs` name to `epochs`.
    '''

    return {
        "epochs": int(n_epochs),
        "imgsz": int(imgsz),
        "batch": int(batch),
        "lr0": float(lr0),
        "patience": int(patience),
        "workers": int(workers),
        "project": str(project),
        "name": str(name),
        "seed": int(seed),
        # Keep checkpoint saving explicit. Ultralytics writes weights/best.pt
        # for the validation-best checkpoint and weights/last.pt for the final epoch.
        "save": True,
        "verbose": False,
    }

## YOLO Warm-Up: Predict Before Training

The official Ultralytics Colab tutorial starts with the shortest useful workflow: install/check the package, run `predict` on a normal image, then move to validation, training, and export. You can open that reference here:

https://colab.research.google.com/github/ultralytics/ultralytics/blob/main/examples/tutorial.ipynb

This notebook uses the same idea, but keeps the warm-up small. First, load a pretrained YOLO model and ask it to predict objects in a familiar image. After that, move to underwater imagery where the pretrained vocabulary and the visual statistics are much less convenient.

The official tutorial currently uses newer YOLO model families. This workshop defaults to `yolo11n.pt` and `yolo11n-seg.pt` because they match the prepared underwater experiments and run quickly in free Colab. You can swap model names later.

In [ ]:
YOLO_WARMUP_SOURCE = "https://ultralytics.com/images/zidane.jpg"
YOLO_WARMUP_CONF = 0.25
YOLO_WARMUP_RESULTS = None

try:
    from ultralytics import YOLO
    import matplotlib.pyplot as plt

    warmup_model = YOLO("yolo11n.pt")
    YOLO_WARMUP_RESULTS = warmup_model.predict(
        source=YOLO_WARMUP_SOURCE,
        imgsz=640,
        conf=YOLO_WARMUP_CONF,
        project=REPO_ROOT / "runs" / "warmup",
        name="predict",
        exist_ok=True,
        verbose=False,
    )

    result = YOLO_WARMUP_RESULTS[0]
    print("CLI equivalent:")
    print(f"yolo predict model=yolo11n.pt source='{YOLO_WARMUP_SOURCE}' conf={YOLO_WARMUP_CONF}")
    print()
    print("Detected classes:", [result.names[int(cls)] for cls in result.boxes.cls.tolist()])

    annotated = result.plot()
    plt.figure(figsize=(8, 5))
    plt.imshow(annotated[..., ::-1])
    plt.axis("off")
    plt.show()

    # Ultralytics may cache downloaded URL sources in the working directory.
    for temporary_name in ["zidane.jpg", "bus.jpg"]:
        temporary_path = Path(temporary_name)
        if temporary_path.exists():
            temporary_path.unlink()
except Exception as exc:
    print("YOLO warm-up prediction skipped. This can happen offline or before dependencies are installed.")
    print(type(exc).__name__, exc)

### YOLO Warm-Up Exercises

Beginner:

- Change `YOLO_WARMUP_CONF` from `0.25` to `0.6`.
- Which detections disappear first?

Intermediate:

- Change `YOLO_WARMUP_SOURCE` to `https://ultralytics.com/images/bus.jpg`.
- Read the printed class names and decide whether the model is solving classification, detection, or segmentation.

Advanced:

- Run the same pretrained model on one underwater validation image before fine-tuning.
- What does the model miss, and what does it hallucinate?

### Master Notes: YOLO Warm-Up

- Raising `YOLO_WARMUP_CONF` from `0.25` to `0.6` should remove lower-confidence detections first; this is a thresholding operation after the model has already scored candidate boxes.
- `bus.jpg` is still detection: the model predicts classes and boxes for multiple objects in one image.
- A generic COCO-pretrained detector often misses underwater organisms because many taxa are outside its label vocabulary, and it may hallucinate familiar COCO classes from shapes or textures.
- This is the same basic pattern as the Ultralytics tutorial: `YOLO(weights)`, then `.predict(...)`, then inspect results.

## Dataset Exploration

You have now seen the images. Next, inspect how the same underlying problem is represented for different machine learning tasks.

The bundle has four useful views:

- `classification_crops`: organism crops grouped by coarse class.
- `yolo_detect_binary`: one-class object detection labels.
- `yolo_segment_binary`: one-class instance segmentation polygon labels.
- `sam3_cached_outputs`: SAM3-like prompt outputs for the fallback lab.

The full source COCO-style annotation file in the local YOLO project has `119,096` images, `280,118` annotations, and `1,897` categories. **COCO** means "Common Objects in Context"; the name comes from a benchmark dataset, but people also use "COCO format" to mean a JSON annotation schema with image records, category records, and annotation records. This tutorial uses a compact subset so you can work in Colab without spending the session on data transfer.

For the live detection and segmentation exercises, the YOLO labels deliberately omit extremely tiny annotations. That keeps the training signal about visible organisms rather than letting dense fragments dominate the classroom metrics. The original COCO subset is still included for discussion and after-session extensions.

The images and annotations are FathomNet-derived. The bundle includes `licenses/attribution.csv`; visual content is subject to contributor-selected Creative Commons terms and is used here for machine-learning training and development.

In [ ]:
from collections import Counter

from scripts.tutorial_data import (
    classification_examples_by_class,
    get_task_paths,
    load_manifest,
    select_coco_category_examples,
    select_yolo_examples,
)
from scripts.tutorial_viz import draw_yolo_boxes, draw_yolo_masks, show_image_grid

detect_paths = get_task_paths("detect", BUNDLE_ROOT)
segment_paths = get_task_paths("segment", BUNDLE_ROOT)
classification_paths = get_task_paths("classification", BUNDLE_ROOT)
coco_paths = get_task_paths("coco", BUNDLE_ROOT)

manifest = load_manifest(BUNDLE_ROOT)
print(json.dumps({
    "name": manifest["name"],
    "workshop_classes": manifest["workshop_classes"],
    "stats": manifest["stats"],
}, indent=2))

### View 1: Full Images With Truth Category Labels

This is a full-image, classification-style view of the COCO-format truth labels. The title above each image lists categories annotated as present in that image. This view hides boxes and masks on purpose, so you can separate the question "what is present?" from the question "where is it?"

In [ ]:
coco_category_examples = select_coco_category_examples(
    coco_paths["json"],
    [detect_paths["root"] / "images" / "train", detect_paths["root"] / "images" / "val"],
    limit=6,
)

def compact_category_title(item, max_categories=2):
    '''Make a short title from ground-truth category names.'''

    from textwrap import wrap

    names = item["category_names"]
    visible = ", ".join(names[:max_categories]) if names else "no named categories"
    if len(names) > max_categories:
        visible += f", +{len(names) - max_categories} more"
    title_lines = wrap(visible, width=34, break_long_words=False) or [visible]
    title_lines.append(f"{item['annotation_count']} annotations")
    return "\n".join(title_lines)

show_image_grid(
    [item["image_path"] for item in coco_category_examples],
    titles=[compact_category_title(item) for item in coco_category_examples],
    columns=3,
    max_images=6,
    figsize=(14, 8),
)

### View 2: Classification Crops With Truth Labels

The training classifier uses organism crops grouped by coarse class. Each crop has one truth label from the folder name. There is still no geometry target here: no center point, no bounding box, no polygon.

In [ ]:
classification_examples = classification_examples_by_class(
    classification_paths["root"],
    split="val",
    examples_per_class=2,
)

show_image_grid(
    [item["image_path"] for item in classification_examples],
    titles=[item["class_name"] for item in classification_examples],
    columns=5,
    max_images=10,
)

### View 3: YOLO Object Detection Truth Labels

Object detection asks for a class and a rectangle for each object. The YOLO detection rows here have the form

`class_id x_center y_center width height`

where the four geometry values are normalized to `[0, 1]` by the image width and height. The examples below are selected because they contain multiple labeled objects.

In [ ]:
import matplotlib.pyplot as plt

detect_multi_examples = select_yolo_examples(
    detect_paths["root"],
    split="val",
    min_instances=2,
    limit=4,
)

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, item in zip(axes.flat, detect_multi_examples):
    draw_yolo_boxes(item["image_path"], item["label_path"], class_names={0: "object"}, ax=ax)
    ax.set_title(f"{Path(item['image_path']).stem[:8]}: {item['instance_count']} objects")
for ax in axes.flat[len(detect_multi_examples):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

### View 4: YOLO Instance Segmentation Truth Labels

Instance segmentation asks for a separate region for each object. In YOLO segmentation format, each row starts with the class id and is followed by normalized polygon coordinates:

`class_id x1 y1 x2 y2 ... xK yK`

The target is more informative than a box, but it is also more expensive to annotate and usually harder to learn.

In [ ]:
segment_multi_examples = select_yolo_examples(
    segment_paths["root"],
    split="val",
    min_instances=2,
    limit=4,
)

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, item in zip(axes.flat, segment_multi_examples):
    draw_yolo_masks(item["image_path"], item["label_path"], class_names={0: "object"}, ax=ax)
    ax.set_title(f"{Path(item['image_path']).stem[:8]}: {item['instance_count']} masks")
for ax in axes.flat[len(segment_multi_examples):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

### Annotation Counts And Category Mix

You have now seen the truth labels visually. The next two cells summarize the same bundle numerically: first the COCO-style categories and annotation counts, then the object-size distribution used by the YOLO detection labels.

In [ ]:
coco = json.loads(coco_paths["json"].read_text())
categories_by_id = {category["id"]: category["name"] for category in coco["categories"]}
annotations_by_image = Counter(annotation["image_id"] for annotation in coco["annotations"])
category_counts = Counter(categories_by_id.get(annotation["category_id"], annotation["category_id"]) for annotation in coco["annotations"])

print("COCO-style subset:")
print(f"  images:      {len(coco['images'])}")
print(f"  annotations: {len(coco['annotations'])}")
print(f"  categories:  {len(coco['categories'])}")
print()
print("Top annotated categories in this small subset:")
for name, count in category_counts.most_common(12):
    print(f"  {name:35s} {count:3d}")
print()
print("Annotations per image:")
print(f"  min={min(annotations_by_image.values())}, max={max(annotations_by_image.values())}, mean={sum(annotations_by_image.values()) / len(annotations_by_image):.2f}")

### Object Size Distribution

This histogram shows how large the detection boxes are as a fraction of image area. It matters because tiny objects are harder to localize, and the tutorial labels intentionally filter out extremely tiny boxes for the live training exercises.

In [ ]:
import matplotlib.pyplot as plt

label_paths = sorted((detect_paths["root"] / "labels" / "train").glob("*.txt")) + sorted((detect_paths["root"] / "labels" / "val").glob("*.txt"))
box_area_fractions = []
instances_per_file = []

for label_path in label_paths:
    rows = [line.split() for line in label_path.read_text().splitlines() if line.strip()]
    instances_per_file.append(len(rows))
    for row in rows:
        _, _, _, width, height = map(float, row)
        box_area_fractions.append(width * height)

print(f"YOLO detection files: {len(label_paths)}")
print(f"YOLO detection instances after tiny-object filtering: {len(box_area_fractions)}")
print(f"instances/image: min={min(instances_per_file)}, max={max(instances_per_file)}, mean={sum(instances_per_file) / len(instances_per_file):.2f}")
print(f"box area fraction: min={min(box_area_fractions):.4f}, median={sorted(box_area_fractions)[len(box_area_fractions)//2]:.4f}, max={max(box_area_fractions):.4f}")

plt.figure(figsize=(7, 3.5))
plt.hist(box_area_fractions, bins=18, edgecolor="black")
plt.xlabel("normalized box area")
plt.ylabel("count")
plt.title("Object sizes in the live YOLO detection labels")
plt.grid(alpha=0.25)
plt.show()

### Dataset Exploration Questions

Beginner:

- Pick one image where the object is visually obvious and one where it is subtle.
- Which source of difficulty is most visible: contrast, scale, clutter, partial view, or taxonomy?

Intermediate:

- Compare the `coco/subset.json` category counts with the binary YOLO labels.
- What information is lost when many biological categories are collapsed into `object`?
- Compare the category-only full-image view with the box and mask views.
- What extra information do you gain when the truth labels include geometry?

Advanced:

- The live YOLO labels drop extremely tiny boxes. Predict how the histogram and mAP would change if those boxes were restored.
- The segmentation labels contain more geometry than boxes. Where would that extra information matter scientifically?

### Master Notes: Dataset Exploration

- Good obvious examples usually have high contrast, a large organism, and a clean background. Subtle examples often involve small objects, partial bodies, or organism-background camouflage.
- The visual views are intentionally distinct: full images with category labels show "what is present," classification crops show one coarse label per crop, detection labels show multiple boxes per full image, and segmentation labels show multiple object-shaped regions per full image.
- Keep this section about truth labels only. Save model predictions for the YOLO warm-up and training/evaluation sections.
- Collapsing many categories into `object` removes biological identity, taxonomy, and ecological meaning, but makes a short detection exercise easier and more stable.
- Restoring tiny boxes would push the area histogram left, increase the number of labels per image, and likely lower short-run mAP for a small model because localization becomes much harder.
- The COCO subset is useful for reasoning about categories and conversion; the prepared YOLO folders are useful for short live training.

# Part 1: Classification From Organism Crops

Classification means predicting one label for an input. This is the easiest entry point because each organism crop has one coarse label. It is still not trivial: underwater crops can be low contrast, partially occluded, visually ambiguous, and taxonomically long-tailed.

Here the image crop is a tensor

$$x \in [0,1]^{H \times W \times 3},$$

and the model produces one score, or logit, for each class. For `K` classes,

$$z=f_\theta(x)\in \mathbb{R}^K,\qquad y\in\{1,\dots,K\}.$$

The softmax turns logits into class probabilities,

$$p_k=\frac{\exp(z_k)}{\sum_j \exp(z_j)},$$

and cross-entropy penalizes the model when it gives low probability to the true class:

$$\ell(z,y)=-\log p_y.$$

The live exercise is to change one training knob, rerun the small model, and interpret whether validation accuracy moved in a meaningful direction.

### Section Bootstrap: Load The Crop Dataset

Start this section by loading the classification paths and printing the split/class counts. If you skipped here from above, this bootstrap cell also restores the common setup variables.

In [ ]:
# Common skip-ahead bootstrap. This block lets a section run even if you did
# not execute the setup cells above.
from pathlib import Path
import json
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/Spiffical/fathomnet-underwater-vision-tutorial"
GITHUB_BRANCH = "main"
PROJECT_DIR_NAME = "fathomnet_underwater_vision_tutorial"
BUNDLE_URL = "https://github.com/Spiffical/fathomnet-underwater-vision-tutorial/raw/main/data/fathomnet_underwater_tutorial_bundle.zip"

if "REPO_ROOT" not in globals():
    def _candidate_roots():
        cwd = Path.cwd().resolve()
        yield cwd
        yield from cwd.parents
        yield Path("/content") / PROJECT_DIR_NAME

    REPO_ROOT = None
    for candidate in _candidate_roots():
        if (candidate / "scripts").exists() and (candidate / "notebooks").exists():
            REPO_ROOT = candidate
            break
    if REPO_ROOT is None and "google.colab" in sys.modules:
        clone_dir = Path("/content") / PROJECT_DIR_NAME
        if not clone_dir.exists():
            subprocess.check_call(["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, f"{GITHUB_REPO_URL}.git", str(clone_dir)])
        REPO_ROOT = clone_dir
    if REPO_ROOT is None:
        raise RuntimeError("Could not find the tutorial repo root. Start Jupyter from the cloned repo.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.tutorial_setup import detect_runtime, download_tutorial_bundle, ensure_dependencies

if "LOCAL_BUNDLE_ZIP" not in globals():
    LOCAL_BUNDLE_ZIP = REPO_ROOT / "data" / "fathomnet_underwater_tutorial_bundle.zip"

ensure_dependencies(install=("google.colab" in sys.modules), extra_pip_args=("--quiet",))

if "BUNDLE_ROOT" not in globals():
    BUNDLE_ROOT = download_tutorial_bundle(
        bundle_url=BUNDLE_URL or None,
        bundle_zip_path=LOCAL_BUNDLE_ZIP if LOCAL_BUNDLE_ZIP.exists() else None,
        output_dir=REPO_ROOT / "data" / "fathomnet_underwater_tutorial_bundle",
    )

if "RUN_LIVE_TRAINING" not in globals():
    RUN_LIVE_TRAINING = bool(detect_runtime().get("has_cuda", False))

if "build_train_args" not in globals():
    def build_train_args(
        *,
        n_epochs=10,
        imgsz=320,
        batch=8,
        lr0=0.001,
        patience=3,
        workers=0,
        project="runs/tutorial",
        name="experiment",
        seed=42,
    ):
        '''Compact fallback copy of the visible helper from the setup section.'''

        return {
            "epochs": int(n_epochs),
            "imgsz": int(imgsz),
            "batch": int(batch),
            "lr0": float(lr0),
            "patience": int(patience),
            "workers": int(workers),
            "project": str(project),
            "name": str(name),
            "seed": int(seed),
            "save": True,
            "verbose": False,
        }


# Section bootstrap: classification
from scripts.tutorial_data import get_task_paths, summarize_classification_dataset
from scripts.tutorial_models import run_classification_lr_trial
from scripts.tutorial_viz import plot_confusion_matrix, plot_training_curves, show_image_grid

CLASSIFY_PATHS = get_task_paths("classification", BUNDLE_ROOT)
CLASSIFY_ROOT = CLASSIFY_PATHS["root"]
CACHED_CLASSIFY_RESULTS = BUNDLE_ROOT / "cached_training" / "classification" / "results.csv"

print(json.dumps(summarize_classification_dataset(CLASSIFY_ROOT), indent=2))

### Inspect The Classification Inputs

Before training, look at one crop from each coarse class. These are the actual inputs for the classifier: one cropped image, one truth label.

In [ ]:
class_examples = []
class_titles = []
for class_dir in sorted((CLASSIFY_ROOT / "train").iterdir()):
    images = sorted(class_dir.glob("*.jpg"))
    if images:
        class_examples.append(images[0])
        class_titles.append(class_dir.name)

show_image_grid(class_examples, titles=class_titles, columns=5)

### Visible Helper: Cross-Entropy

This is a small, editable version of the loss behind multi-class classification. It is not meant to replace PyTorch; it is here so the math is visible.

In [ ]:
def softmax_cross_entropy_from_logits(logits, true_class):
    '''Compute -log softmax(logits)[true_class] in a numerically stable way.'''

    import math

    max_logit = max(logits)
    shifted = [value - max_logit for value in logits]
    normalizer = sum(math.exp(value) for value in shifted)
    log_probability = shifted[true_class] - math.log(normalizer)
    return -log_probability


example_logits = [1.2, -0.4, 0.7, 2.1, 0.0]
print("loss if the true class is index 3:", softmax_cross_entropy_from_logits(example_logits, 3))
print("loss if the true class is index 1:", softmax_cross_entropy_from_logits(example_logits, 1))

### Train Or Load A Small Classification Run

Now move from the hand-computed loss to an actual model. This cell trains `yolo11n-cls.pt` when a GPU is available. If not, it loads the cached classification curve so the interpretation exercise still works.

Focus on the training arguments: `n_epochs`, `imgsz`, `batch`, and `lr0`. Those are the knobs you will modify in the exercises. When live training runs, Ultralytics saves the validation-best checkpoint at `weights/best.pt`.

In [ ]:
CLASSIFY_N_EPOCHS = 10
CLASSIFY_ARGS = build_train_args(
    n_epochs=CLASSIFY_N_EPOCHS,
    imgsz=224,
    batch=16,
    lr0=0.001,
    project=REPO_ROOT / "runs" / "classification",
    name="classification_default",
)

# Documentation checkpoint:
# 1. Open https://docs.ultralytics.com/tasks/classify/
# 2. Find how a pretrained classification model is loaded.
# 3. Compare that with the general training pattern at https://docs.ultralytics.com/modes/train/
#
# Your live exercise is to modify the model name or train arguments below.
classification_training = None
classification_save_dir = None
classification_best_model_path = None
if RUN_LIVE_TRAINING:
    from ultralytics import YOLO

    classification_model = YOLO("yolo11n-cls.pt")
    classification_training = classification_model.train(data=str(CLASSIFY_ROOT), **CLASSIFY_ARGS)
    classification_save_dir = getattr(classification_training, "save_dir", None)
    if classification_save_dir is None and getattr(classification_model, "trainer", None) is not None:
        classification_save_dir = getattr(classification_model.trainer, "save_dir", None)
    print("Live classification training finished.")
    candidate_best = Path(classification_save_dir) / "weights" / "best.pt" if classification_save_dir is not None else None
    if candidate_best is not None and candidate_best.exists():
        classification_best_model_path = candidate_best
        print(f"Best validation checkpoint: {classification_best_model_path}")
else:
    print("No GPU detected. Using cached classification training curve.")

classification_results_csv = (
    Path(classification_save_dir) / "results.csv"
    if classification_save_dir is not None
    else CACHED_CLASSIFY_RESULTS
)
if not classification_results_csv.exists():
    classification_results_csv = CACHED_CLASSIFY_RESULTS

plot_training_curves(classification_results_csv, title="Classification learning curve")

### Mini-Lab: Learning Rate As An Optimization Knob

The learning rate controls the scale of each gradient step. In a small workshop run, you usually cannot prove one setting is globally best. You can still learn to recognize three useful patterns:

- too small: the curve barely moves;
- useful: validation accuracy improves without wild instability;
- too large: loss or validation accuracy jumps around or degrades.

Try one or more learning rates from `lr0 = 1e-4`, `1e-3`, and `1e-2`. The cached curves give you a baseline comparison, and a live GPU run lets you see how much the result changes from one run to the next.

### Master Notes: Learning Rate Lab

- `1e-4` is likely to look slow in a one-epoch or two-epoch workshop run.
- `1e-3` is a reasonable default for this tiny classification fine-tuning exercise.
- `1e-2` may improve quickly or become unstable, depending on batch order and initialization.
- The teaching point is not that one run proves the best learning rate. It is that optimization behavior is visible in curves, and validation accuracy is a noisy estimate on this small split.

In [ ]:
RUN_CLASSIFICATION_LR_LAB = False
LR_VALUES = [1e-4, 1e-3, 1e-2]

if RUN_CLASSIFICATION_LR_LAB and RUN_LIVE_TRAINING:
    lr_rows = [
        run_classification_lr_trial(
            CLASSIFY_ROOT,
            build_train_args,
            lr0=lr0,
            repo_root=REPO_ROOT,
            n_epochs=1,
        )
        for lr0 in LR_VALUES
    ]
    for row in lr_rows:
        print(row)
        plot_training_curves(row["results_csv"], title=f"classification lr0={row['lr0']}")
else:
    print("Learning-rate lab is ready.")
    print("Set RUN_CLASSIFICATION_LR_LAB = True on a GPU to run these trials live.")
    print("Suggested lr0 values:", LR_VALUES)

### Confusion Matrix Reading Practice

The next cell uses a small discussion matrix rather than requiring live predictions. Read it row by row: each row is a true class, and each column is the predicted class. The goal is to connect metric summaries to specific biological or visual confusions.

In [ ]:
# A small confusion matrix for discussion. Replace this with predictions from
# your live run if you want to make the exercise more advanced.
classes = ["echinoderm", "fish", "gelatinous", "crustacean", "sponge_coral"]
toy_confusion = [
    [8, 0, 1, 0, 1],
    [0, 6, 1, 1, 0],
    [1, 0, 7, 0, 2],
    [0, 2, 0, 5, 1],
    [1, 0, 2, 0, 8],
]
plot_confusion_matrix(toy_confusion, classes, normalize=True, title="Discussion matrix: normalized by true class")

### Classification Exercises

Beginner:

- Change `CLASSIFY_N_EPOCHS` from `10` to another value, then rerun the training cell.
- Run or reason through one learning-rate trial.
- Decide whether the validation curve is improving, noisy, or overfitting.

Intermediate:

- Use the Ultralytics classification docs to find the direct `YOLO(...).train(...)` pattern for classification.
- Compare one `lr0` result with another `lr0` result.
- Change `imgsz` from `224` to `320`, then decide whether the extra computation was worth it.
- Pick one class and inspect three crops that seem visually ambiguous.

Advanced:

- Design a fair pretrained-vs-random-initialization comparison. What would you keep fixed?
- Replace the toy confusion matrix with predictions from your live model.
- Which two classes are most confusable, and what visual ambiguity might explain it?
- Propose a better class grouping for this small crop dataset.

### Master Notes: Classification

- The direct Ultralytics pattern is `from ultralytics import YOLO`, `model = YOLO("yolo11n-cls.pt")`, then `model.train(data=str(CLASSIFY_ROOT), ...)`.
- Increasing `n_epochs` should usually improve the tiny validation run at first, but the curve can be noisy because the validation set is intentionally small.
- Increasing `lr0` by `10x` may converge faster or destabilize; decreasing it by `10x` may make one or two epochs look almost unchanged.
- In the toy confusion matrix, `gelatinous` and `sponge_coral` are mutually confusable, and `crustacean` can leak into `fish`. Good discussion targets are transparency, partial views, posture, and texture.

# Part 2: Binary Object Detection With YOLO

Detection asks for *where* the animals or biological structures are, not only what crop class they belong to. In this section the target is a **bounding box**, or a rectangle that encloses an object.

Instead of one label per crop, the output is a set of boxes:

$$\{(c_i,b_i,s_i)\}_{i=1}^m,\qquad b_i=(c_x,c_y,w,h).$$

Here `c_i` is the class, `b_i` is a normalized box, and `s_i` is the confidence score. YOLO stores `c_x`, `c_y`, `w`, and `h` in `[0,1]` relative to image width and height.

The workshop bundle uses a binary **object** target and filters very small boxes. This makes the first detector behave like a useful mathematical object during a short tutorial: you can see IoU, confidence thresholds, precision, recall, and AP move for interpretable reasons.

During training, the model parameters are chosen by minimizing empirical risk on the training split:

$$
\hat{\theta}\in\arg\min_\theta \frac{1}{n}\sum_{i=1}^n \ell(f_\theta(x_i), y_i).
$$

The validation split is the quick estimate of whether those parameters are useful outside the examples that supplied the gradients.

For one ground-truth box `A` and one predicted box `B`, the intersection-over-union is

$$\operatorname{IoU}(A,B)=\frac{|A \cap B|}{|A \cup B|}.$$

Precision and recall depend on a confidence threshold:

$$\mathrm{precision}=\frac{TP}{TP+FP}, \qquad \mathrm{recall}=\frac{TP}{TP+FN}.$$

Average precision, or **AP**, summarizes the precision-recall curve as that threshold changes. **mAP** means mean average precision. In YOLO reports, `mAP50` uses an IoU threshold of `0.50`, while `mAP50-95` averages over several IoU thresholds from `0.50` to `0.95`, making it a stricter localization metric.

### Mini-Lab: Matching Predictions To Labels

Detection metrics are built from a matching rule. A predicted box becomes a true positive only if it has enough overlap with an unmatched ground-truth box. Everything else is a false positive, and every unmatched ground-truth box is a false negative.

Use this toy example before looking at YOLO outputs. It is small enough to calculate by hand.

### Master Notes: Matching Predictions

- At `conf >= 0.25`, the toy example keeps all three predictions: one true positive and two false positives, so precision is `1/3` and recall is `1`.
- At `conf >= 0.5`, it keeps the first two predictions: one true positive and one false positive, so precision is `1/2` and recall is `1`.
- At `conf >= 0.8`, it keeps only the best prediction: one true positive, zero false positives, so precision is `1` and recall is `1`.
- In real multi-object AP, matching is repeated over many objects and thresholds, but this one-object case is the core logic.

In [ ]:
def box_iou_xyxy(box_a, box_b):
    '''IoU for boxes in [x_min, y_min, x_max, y_max] pixel coordinates.'''

    ax0, ay0, ax1, ay1 = box_a
    bx0, by0, bx1, by1 = box_b
    inter_x0 = max(ax0, bx0)
    inter_y0 = max(ay0, by0)
    inter_x1 = min(ax1, bx1)
    inter_y1 = min(ay1, by1)
    inter_width = max(0, inter_x1 - inter_x0)
    inter_height = max(0, inter_y1 - inter_y0)
    intersection = inter_width * inter_height
    area_a = max(0, ax1 - ax0) * max(0, ay1 - ay0)
    area_b = max(0, bx1 - bx0) * max(0, by1 - by0)
    union = area_a + area_b - intersection
    return intersection / union if union else 0.0


def one_object_precision_recall(predictions, ground_truth_box, *, conf_threshold=0.25, iou_threshold=0.5):
    '''Simplified precision/recall for one ground-truth object.

    `predictions` is a list of dictionaries with `box` and `score`.
    The highest-score prediction above the threshold that overlaps enough gets
    matched as the true positive. Other predictions above threshold are false positives.
    '''

    kept = [prediction for prediction in predictions if prediction["score"] >= conf_threshold]
    kept = sorted(kept, key=lambda prediction: prediction["score"], reverse=True)
    matched = False
    true_positives = 0
    false_positives = 0

    for prediction in kept:
        iou = box_iou_xyxy(prediction["box"], ground_truth_box)
        prediction["iou"] = iou
        if iou >= iou_threshold and not matched:
            true_positives += 1
            matched = True
        else:
            false_positives += 1

    false_negatives = 0 if matched else 1
    precision = true_positives / (true_positives + false_positives) if kept else 0.0
    recall = true_positives / (true_positives + false_negatives)
    return {
        "kept_predictions": kept,
        "tp": true_positives,
        "fp": false_positives,
        "fn": false_negatives,
        "precision": precision,
        "recall": recall,
    }


toy_ground_truth = [10, 10, 50, 50]
toy_predictions = [
    {"box": [12, 12, 48, 48], "score": 0.92},
    {"box": [55, 12, 90, 45], "score": 0.70},
    {"box": [8, 8, 35, 35], "score": 0.35},
]

for threshold in [0.25, 0.5, 0.8]:
    summary = one_object_precision_recall(toy_predictions, toy_ground_truth, conf_threshold=threshold)
    print(f"conf >= {threshold}:", {key: summary[key] for key in ["tp", "fp", "fn", "precision", "recall"]})

### Section Bootstrap: Load The Detection Dataset

The toy example above shows the metric logic. Now switch to the real YOLO detection dataset. This bootstrap cell loads the dataset YAML, validates image/label pairing, and defines the cached result path used when live training is unavailable.

In [ ]:
# Common skip-ahead bootstrap. This block lets a section run even if you did
# not execute the setup cells above.
from pathlib import Path
import json
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/Spiffical/fathomnet-underwater-vision-tutorial"
GITHUB_BRANCH = "main"
PROJECT_DIR_NAME = "fathomnet_underwater_vision_tutorial"
BUNDLE_URL = "https://github.com/Spiffical/fathomnet-underwater-vision-tutorial/raw/main/data/fathomnet_underwater_tutorial_bundle.zip"

if "REPO_ROOT" not in globals():
    def _candidate_roots():
        cwd = Path.cwd().resolve()
        yield cwd
        yield from cwd.parents
        yield Path("/content") / PROJECT_DIR_NAME

    REPO_ROOT = None
    for candidate in _candidate_roots():
        if (candidate / "scripts").exists() and (candidate / "notebooks").exists():
            REPO_ROOT = candidate
            break
    if REPO_ROOT is None and "google.colab" in sys.modules:
        clone_dir = Path("/content") / PROJECT_DIR_NAME
        if not clone_dir.exists():
            subprocess.check_call(["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, f"{GITHUB_REPO_URL}.git", str(clone_dir)])
        REPO_ROOT = clone_dir
    if REPO_ROOT is None:
        raise RuntimeError("Could not find the tutorial repo root. Start Jupyter from the cloned repo.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.tutorial_setup import detect_runtime, download_tutorial_bundle, ensure_dependencies

if "LOCAL_BUNDLE_ZIP" not in globals():
    LOCAL_BUNDLE_ZIP = REPO_ROOT / "data" / "fathomnet_underwater_tutorial_bundle.zip"

ensure_dependencies(install=("google.colab" in sys.modules), extra_pip_args=("--quiet",))

if "BUNDLE_ROOT" not in globals():
    BUNDLE_ROOT = download_tutorial_bundle(
        bundle_url=BUNDLE_URL or None,
        bundle_zip_path=LOCAL_BUNDLE_ZIP if LOCAL_BUNDLE_ZIP.exists() else None,
        output_dir=REPO_ROOT / "data" / "fathomnet_underwater_tutorial_bundle",
    )

if "RUN_LIVE_TRAINING" not in globals():
    RUN_LIVE_TRAINING = bool(detect_runtime().get("has_cuda", False))

if "build_train_args" not in globals():
    def build_train_args(
        *,
        n_epochs=10,
        imgsz=320,
        batch=8,
        lr0=0.001,
        patience=3,
        workers=0,
        project="runs/tutorial",
        name="experiment",
        seed=42,
    ):
        '''Compact fallback copy of the visible helper from the setup section.'''

        return {
            "epochs": int(n_epochs),
            "imgsz": int(imgsz),
            "batch": int(batch),
            "lr0": float(lr0),
            "patience": int(patience),
            "workers": int(workers),
            "project": str(project),
            "name": str(name),
            "seed": int(seed),
            "save": True,
            "verbose": False,
        }


# Section bootstrap: detection
from scripts.tutorial_data import get_task_paths, make_tiny_detection_dataset, select_yolo_examples, validate_yolo_dataset
from scripts.tutorial_models import prediction_count_at_threshold
from scripts.tutorial_viz import draw_yolo_boxes, plot_training_curves

DETECT_PATHS = get_task_paths("detect", BUNDLE_ROOT)
DETECT_ROOT = DETECT_PATHS["root"]
DETECT_YAML = DETECT_PATHS["yaml"]
CACHED_DETECT_RESULTS = BUNDLE_ROOT / "cached_training" / "detection" / "results.csv"

print(json.dumps(validate_yolo_dataset(DETECT_YAML, task="detect"), indent=2)[:3000])

### Detection Truth Example: Several Objects In One Image

This plot shows one validation image from the YOLO detection dataset. The green rectangles are **ground-truth labels**, not model predictions. Each row in the matching `.txt` label file says: class id, normalized box center, normalized box width, and normalized box height.

The example is selected to contain multiple labeled objects so the detection task is visually clear.

In [ ]:
import matplotlib.pyplot as plt

PREFERRED_DETECT_EXAMPLE_STEM = "eb7661e6-ded7-49dc-b335-0e7dfbb5d775"
detect_candidates = select_yolo_examples(DETECT_ROOT, split="val", min_instances=2, limit=6)
detect_example = next(
    (
        item
        for item in detect_candidates
        if item["image_path"].stem == PREFERRED_DETECT_EXAMPLE_STEM
    ),
    detect_candidates[0],
)
first_detect_label = detect_example["label_path"]
first_detect_image = detect_example["image_path"]
DETECT_EXAMPLE_STEM = first_detect_image.stem

print(f"Showing {DETECT_EXAMPLE_STEM}: {detect_example['instance_count']} truth boxes")
ax = draw_yolo_boxes(first_detect_image, first_detect_label, class_names={0: "object"})
ax.set_title(f"Detection truth labels: {detect_example['instance_count']} objects")
plt.show()

### Train Or Load A Small Detection Run

Now we train a binary object detector. The model starts from `yolo11n.pt`, an Ultralytics YOLO detection checkpoint, and the dataset comes from `DETECT_YAML`.

If a GPU is available, this cell runs a short fine-tuning job. Otherwise, it loads a cached training curve. Either way, the output you should inspect is the same: precision, recall, `mAP50`, and `mAP50-95`. For a live run, the best validation checkpoint is saved as `weights/best.pt`.

In [ ]:
DETECT_N_EPOCHS = 10
DETECT_ARGS = build_train_args(
    n_epochs=DETECT_N_EPOCHS,
    imgsz=320,
    batch=8,
    lr0=0.001,
    project=REPO_ROOT / "runs" / "detect",
    name="detect_default",
)

# Documentation checkpoint:
# 1. Open https://docs.ultralytics.com/modes/train/
# 2. Find the Python example that loads a pretrained model.
# 3. Find where `data=...`, `epochs=...`, and `imgsz=...` enter `model.train(...)`.
#
# Your live exercise is to understand and modify this small reference block.
# It uses direct Ultralytics code rather than a tutorial utility wrapper.
detection_training = None
detection_save_dir = None
detection_best_model_path = None
if RUN_LIVE_TRAINING:
    from ultralytics import YOLO

    detection_model = YOLO("yolo11n.pt")
    detection_training = detection_model.train(data=str(DETECT_YAML), **DETECT_ARGS)
    detection_save_dir = getattr(detection_training, "save_dir", None)
    if detection_save_dir is None and getattr(detection_model, "trainer", None) is not None:
        detection_save_dir = getattr(detection_model.trainer, "save_dir", None)
    print("Live detection training finished.")
    candidate_best = Path(detection_save_dir) / "weights" / "best.pt" if detection_save_dir is not None else None
    if candidate_best is not None and candidate_best.exists():
        detection_best_model_path = candidate_best
        print(f"Best validation checkpoint: {detection_best_model_path}")
else:
    print("No GPU detected. Using cached detection training curve.")

detection_results_csv = (
    Path(detection_save_dir) / "results.csv"
    if detection_save_dir is not None
    else CACHED_DETECT_RESULTS
)
if not detection_results_csv.exists():
    detection_results_csv = CACHED_DETECT_RESULTS

plot_training_curves(
    detection_results_csv,
    metric_columns=["metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"],
    title="Detection metrics",
)

### Mini-Lab: Confidence Threshold Sweep

Training gives the model parameters. The confidence threshold is a decision rule you choose after the model scores candidate detections.

Sweep a few thresholds on the same underwater image. Your job is to describe the tradeoff in plain language before you look at the metric names:

- low threshold: more detections, more possible false positives;
- high threshold: fewer detections, more possible missed objects.

### Master Notes: Threshold Sweep

- Lower thresholds tend to increase the number of detections and increase recall opportunities.
- Higher thresholds tend to suppress weak boxes, which often improves precision but can remove real objects.
- The model weights do not change during this lab. Only the decision rule changes.
- If the generic COCO detector finds little in underwater images, that is itself a domain-shift result.

In [ ]:
THRESHOLD_SWEEP_VALUES = [0.10, 0.25, 0.50, 0.80]
threshold_sweep_rows = []

try:
    from ultralytics import YOLO
    import matplotlib.pyplot as plt

    threshold_weights = (
        detection_best_model_path
        if detection_best_model_path is not None and detection_best_model_path.exists()
        else "yolo11n.pt"
    )
    threshold_model = YOLO(str(threshold_weights))
    threshold_image = first_detect_image
    threshold_sweep_rows = [
        prediction_count_at_threshold(threshold_model, threshold_image, conf=value, repo_root=REPO_ROOT)
        for value in THRESHOLD_SWEEP_VALUES
    ]

    for row in threshold_sweep_rows:
        print(f"conf={row['conf']:.2f}: detections={row['detections']}, mean score={row['mean_score']:.3f}")

    fig, axes = plt.subplots(1, len(threshold_sweep_rows), figsize=(4 * len(threshold_sweep_rows), 4))
    for ax, row in zip(axes, threshold_sweep_rows):
        ax.imshow(row["result"].plot()[..., ::-1])
        ax.set_title(f"conf >= {row['conf']}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print("Threshold sweep skipped. You can run it after Ultralytics is available.")
    print(type(exc).__name__, exc)

### Mini-Lab: Overfit A Tiny Dataset

A powerful debugging move is to ask whether the model can memorize a tiny training set. If it cannot overfit 8-12 examples, something may be wrong with the labels, model wiring, optimization settings, or data path.

This cell prepares the tiny dataset for you. Training is off by default because it is a live GPU exercise.

### Master Notes: Tiny Overfit

- A successful tiny-overfit run should drive training loss down sharply.
- Validation mAP may remain noisy because the validation set is also tiny and may contain different-looking objects.
- If training loss does not drop, first suspect data paths, label format, learning rate, batch size, or whether the labels actually align with the images.
- This is a debugging test, not a final model-quality test.

In [ ]:
RUN_TINY_OVERFIT_LAB = False
tiny_detect_yaml = make_tiny_detection_dataset(
    DETECT_ROOT,
    REPO_ROOT / "tmp" / "tiny_detection_overfit",
    train_images=8,
    val_images=8,
)
print(f"Tiny overfit dataset: {tiny_detect_yaml}")

if RUN_TINY_OVERFIT_LAB and RUN_LIVE_TRAINING:
    from ultralytics import YOLO

    TINY_OVERFIT_N_EPOCHS = 10
    tiny_args = build_train_args(
        n_epochs=TINY_OVERFIT_N_EPOCHS,
        imgsz=320,
        batch=4,
        lr0=0.003,
        patience=20,
        project=REPO_ROOT / "runs" / "tiny_overfit",
        name="detect_8_images",
    )
    tiny_model = YOLO("yolo11n.pt")
    tiny_result = tiny_model.train(data=str(tiny_detect_yaml), **tiny_args)
    tiny_save_dir = getattr(tiny_result, "save_dir", None) or getattr(tiny_model.trainer, "save_dir", None)
    tiny_best_model_path = Path(tiny_save_dir) / "weights" / "best.pt"
    if tiny_best_model_path.exists():
        print(f"Best validation checkpoint: {tiny_best_model_path}")
    plot_training_curves(
        Path(tiny_save_dir) / "results.csv",
        metric_columns=["train/box_loss", "val/box_loss", "metrics/mAP50(B)"],
        title="Tiny-dataset overfit check",
    )
else:
    print("Tiny overfit lab is ready. Set RUN_TINY_OVERFIT_LAB = True on a GPU.")
    print("Success criterion: training loss drops strongly; validation may or may not improve.")

### Advanced Exercise: Start From FathomNet Megalodon

The detector above starts from a general YOLO checkpoint. A stronger underwater baseline is [FathomNet Megalodon](https://huggingface.co/FathomNet/megalodon), a YOLOv8x object detector fine-tuned by MBARI/FathomNet to detect one class: `object`. The model card reports that it was trained from publicly available FathomNet localizations and is meant for post-processing underwater images and video.

This is an advanced path because the checkpoint is larger than `yolo11n.pt`, and fine-tuning it can be slow on a small free GPU. The key question is worth the trouble:

> Does a domain-specific FathomNet checkpoint give you a better starting point than a generic image checkpoint?

Useful references:

- FathomNet Megalodon model card: https://huggingface.co/FathomNet/megalodon
- Hugging Face Hub download docs: https://huggingface.co/docs/huggingface_hub/guides/download
- Ultralytics prediction mode: https://docs.ultralytics.com/modes/predict/
- Ultralytics training and fine-tuning mode: https://docs.ultralytics.com/modes/train/

You will not get the full solution here. The cells below give constants, checks, and hints. Your job is to use the documentation to fill in the missing pieces:

1. download the checkpoint from Hugging Face;
2. load it as a YOLO model;
3. run prediction on `first_detect_image`;
4. optionally fine-tune it on `DETECT_YAML`.

### Master Notes: Megalodon

- This should be framed as an advanced optional path. The checkpoint is much larger than `yolo11n.pt`, and YOLOv8x fine-tuning can be slow on small GPUs.
- The useful comparison is not only metric-vs-metric. Ask whether Megalodon finds underwater object-like regions that generic COCO weights miss.
- Keep `n_epochs` modest at first. The first fine-tuning question is whether the run is wired correctly and whether early validation behavior looks plausible.
- Because Megalodon is already a one-class FathomNet detector, it is a good match for the tutorial's binary `object` labels.
- The participant notebook intentionally gives hints rather than complete code here. For a quick solution, use `hf_hub_download(...)`, then `YOLO(str(path)).predict(...)`, then `YOLO(str(path)).train(data=str(DETECT_YAML), ...)`.

In [ ]:
RUN_MEGALODON_ADVANCED = False

MEGALODON_REPO_ID = "FathomNet/megalodon"
MEGALODON_FILENAME = "mbari-megalodon-yolov8x.pt"
MEGALODON_MODEL_PATH = None

if RUN_MEGALODON_ADVANCED:
    # TODO:
    # 1. Import the Hugging Face download helper.
    # 2. Download `MEGALODON_FILENAME` from `MEGALODON_REPO_ID`.
    # 3. Store the returned local path in `MEGALODON_MODEL_PATH`.
    #
    # Hints:
    # - The helper is called `hf_hub_download`.
    # - Use `cache_dir=REPO_ROOT / ".cache" / "huggingface"` if you want the
    #   checkpoint to stay inside the tutorial repo.
    print("TODO: download the Megalodon checkpoint and set MEGALODON_MODEL_PATH.")
else:
    print("Advanced Megalodon path is off by default.")
    print("Set RUN_MEGALODON_ADVANCED = True to download the FathomNet checkpoint.")

#### Prediction Task

After the checkpoint path exists, use the Ultralytics prediction documentation to run Megalodon on `first_detect_image`. Compare the predicted boxes with the ground-truth boxes from the earlier detection-truth cell.

In [ ]:
RUN_MEGALODON_PREDICT = False

if RUN_MEGALODON_PREDICT and MEGALODON_MODEL_PATH is not None:
    # TODO:
    # 1. Import YOLO from ultralytics.
    # 2. Load `MEGALODON_MODEL_PATH`.
    # 3. Run `.predict(...)` on `first_detect_image`.
    # 4. Plot the first result with `result.plot()`.
    #
    # Hints:
    # - Start with the Ultralytics prediction docs.
    # - Use a moderate image size first, for example `imgsz=640`.
    # - Compare the output with the truth boxes shown above.
    print("TODO: load Megalodon with YOLO(...) and run prediction.")
else:
    print("Prediction exercise ready.")
    print("Set RUN_MEGALODON_PREDICT = True after you have downloaded the checkpoint.")

#### Optional Fine-Tuning Scaffold

Fine-tuning means continuing optimization from an existing checkpoint instead of starting from generic weights. For this small tutorial dataset, use a small learning rate and very few epochs first. Your goal is not to produce a publishable model in one run; your goal is to learn whether the domain-specific starting point changes the early training behavior.

Before running the cell, read the Ultralytics training examples and identify:

- where the checkpoint path enters `YOLO(...)`;
- where the dataset YAML enters `model.train(data=...)`;
- which arguments control image size, batch size, learning rate, and epoch count.

In [ ]:
RUN_MEGALODON_FINE_TUNE = False

if RUN_MEGALODON_ADVANCED and RUN_MEGALODON_FINE_TUNE and RUN_LIVE_TRAINING and MEGALODON_MODEL_PATH is not None:
    # TODO:
    # 1. Build a small training-argument dictionary.
    # 2. Load the Megalodon checkpoint with YOLO(...).
    # 3. Call `.train(data=str(DETECT_YAML), ...)`.
    # 4. Plot the resulting `results.csv` with `plot_training_curves`.
    #
    # Hints:
    # - Keep this small at first: a modest `n_epochs`, batch 1-2, and a small `lr0`.
    # - The checkpoint is larger than YOLO11n, so memory is the limiting resource.
    # - Use the generic detection training cell above as a pattern, but do not
    #   copy it blindly; check each argument against the Ultralytics docs.
    print("TODO: fine-tune Megalodon on DETECT_YAML and plot the metrics.")
else:
    print("Fine-tuning scaffold is ready.")
    print("Set RUN_MEGALODON_ADVANCED = True, RUN_MEGALODON_FINE_TUNE = True, and use a GPU.")

### Exercise: COCO Boxes To YOLO Boxes

The bundle already includes YOLO labels so you can spend most of the session on modeling. The real conversion step is deliberately left as an exercise.

COCO stores a bounding box as

```text
[x_min, y_min, width, height]
```

in pixel coordinates. YOLO detection stores one row as

```text
class_id center_x center_y width height
```

with all four geometry values normalized to `[0,1]`.

In [ ]:
def coco_bbox_to_yolo_exercise(coco_bbox, image_width, image_height):
    '''Convert one COCO [x_min, y_min, width, height] box to YOLO geometry.

    Return `(center_x, center_y, width, height)`, normalized by image size.
    '''

    x_min, y_min, box_width, box_height = coco_bbox
    center_x = (x_min + box_width / 2) / image_width
    center_y = (y_min + box_height / 2) / image_height
    width = box_width / image_width
    height = box_height / image_height
    return (center_x, center_y, width, height)

example_coco_bbox = [20, 10, 40, 30]
candidate_box = coco_bbox_to_yolo_exercise(example_coco_bbox, image_width=100, image_height=100)

print("Your YOLO box:", candidate_box)
assert candidate_box == (0.4, 0.25, 0.4, 0.3)
assert len(candidate_box) == 4
assert all(0 <= value <= 1 for value in candidate_box)

### Mini-Lab: Error Taxonomy

Metrics tell you *how much* error there is. A failure taxonomy tells you *what kind* of error it is. For underwater imagery, common categories are:

- good prediction;
- missed small object;
- poor localization;
- duplicate detection;
- object-like background texture;
- ambiguous or incomplete label.

Look at a few predictions and assign one category to each. This is how you turn a metric into a modeling decision.

### Master Notes: Error Taxonomy

- The most productive discussion is usually not "the model is bad," but "which failure mode is dominant?"
- Missed small objects suggest resolution, label filtering, or data quantity issues.
- Object-like background false positives suggest thresholding, hard-negative examples, or stronger domain-specific pretraining.
- Poor localization suggests label geometry, image size, or training duration.

In [ ]:
ERROR_CATEGORIES = [
    "good prediction",
    "missed small object",
    "poor localization",
    "duplicate detection",
    "object-like background texture",
    "ambiguous or incomplete label",
]

print("Use these categories while inspecting predictions:")
for index, category in enumerate(ERROR_CATEGORIES, start=1):
    print(f"{index}. {category}")

print()
print("Suggested workflow:")
print("1. Pick two images from the threshold sweep.")
print("2. Compare predictions against the green ground-truth boxes.")
print("3. Assign one error category and write one sentence explaining the likely cause.")

### Detection Exercises

Beginner:

- Use the Ultralytics training guide to identify the two lines that load a model and start training.
- Run the confidence threshold sweep and describe what changes from `0.10` to `0.80`.
- Complete the toy matching exercise: what is TP, FP, and FN at each confidence threshold?

Intermediate:

- Change `imgsz` from `320` to `416`.
- Change `DETECT_N_EPOCHS`, `lr0`, or `batch`, then compare `mAP50` and recall.
- Complete `coco_bbox_to_yolo_exercise(...)` above.
- Use the error taxonomy on two predicted images.

Advanced:

- Turn on `RUN_TINY_OVERFIT_LAB` and check whether the model can memorize 8 training images.
- Turn on the Megalodon advanced path, compare its predictions with `yolo11n.pt`, and decide whether the domain-specific checkpoint changes the failure modes.
- Fine-tune Megalodon for one or two epochs, then compare its early metrics with the generic checkpoint run.
- Explain how the precision-recall curve would move if the model became more conservative.
- Sketch how you would convert the whole `coco/subset.json` file into YOLO detection labels.

### Master Notes: Detection

- The two essential training lines are `model = YOLO("yolo11n.pt")` and `model.train(data=str(DETECT_YAML), **DETECT_ARGS)`.
- COCO `[x_min, y_min, width, height]` maps to YOLO `(x_min + width/2)/W`, `(y_min + height/2)/H`, `width/W`, `height/H`.
- A more conservative confidence threshold usually raises precision and lowers recall.
- Raising `imgsz` can help small objects because the resized image preserves more spatial detail, but it costs more memory and time.
- A whole-file COCO converter needs to group annotations by image, map category ids to contiguous YOLO class ids, normalize geometry by each image size, skip invalid boxes, and write one `.txt` label file per image.

# Part 3: Instance Segmentation With YOLO

Segmentation means predicting regions or pixels. **Instance segmentation** upgrades a rectangle to a separate shape for each object instance. A mask can be viewed as a set of pixels

$$M \subseteq \Omega,\qquad \Omega=\{1,\dots,H\}\times\{1,\dots,W\}.$$

For multiple objects, the model predicts a set of class-mask-score triples:

$$\{(c_i,M_i,s_i)\}_{i=1}^m.$$

As in the detection section, the live labels focus on larger visible instances. The omitted tiny polygons are a useful reminder that label design is part of the modeling problem, not a boring preprocessing footnote.

The same IoU idea applies:

$$\operatorname{maskIoU}(M,N)=\frac{|M \cap N|}{|M \cup N|}.$$

In the YOLO segmentation text format, one row is

```text
class x1 y1 x2 y2 ... xn yn
```

where polygon coordinates are normalized to `[0,1]`.

### Section Bootstrap: Load The Segmentation Dataset

Start the segmentation section by loading the YOLO segmentation YAML and validating its labels. This is the same pattern as detection, but each label row now stores a polygon instead of a rectangle.

In [ ]:
# Common skip-ahead bootstrap. This block lets a section run even if you did
# not execute the setup cells above.
from pathlib import Path
import json
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/Spiffical/fathomnet-underwater-vision-tutorial"
GITHUB_BRANCH = "main"
PROJECT_DIR_NAME = "fathomnet_underwater_vision_tutorial"
BUNDLE_URL = "https://github.com/Spiffical/fathomnet-underwater-vision-tutorial/raw/main/data/fathomnet_underwater_tutorial_bundle.zip"

if "REPO_ROOT" not in globals():
    def _candidate_roots():
        cwd = Path.cwd().resolve()
        yield cwd
        yield from cwd.parents
        yield Path("/content") / PROJECT_DIR_NAME

    REPO_ROOT = None
    for candidate in _candidate_roots():
        if (candidate / "scripts").exists() and (candidate / "notebooks").exists():
            REPO_ROOT = candidate
            break
    if REPO_ROOT is None and "google.colab" in sys.modules:
        clone_dir = Path("/content") / PROJECT_DIR_NAME
        if not clone_dir.exists():
            subprocess.check_call(["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, f"{GITHUB_REPO_URL}.git", str(clone_dir)])
        REPO_ROOT = clone_dir
    if REPO_ROOT is None:
        raise RuntimeError("Could not find the tutorial repo root. Start Jupyter from the cloned repo.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.tutorial_setup import detect_runtime, download_tutorial_bundle, ensure_dependencies

if "LOCAL_BUNDLE_ZIP" not in globals():
    LOCAL_BUNDLE_ZIP = REPO_ROOT / "data" / "fathomnet_underwater_tutorial_bundle.zip"

ensure_dependencies(install=("google.colab" in sys.modules), extra_pip_args=("--quiet",))

if "BUNDLE_ROOT" not in globals():
    BUNDLE_ROOT = download_tutorial_bundle(
        bundle_url=BUNDLE_URL or None,
        bundle_zip_path=LOCAL_BUNDLE_ZIP if LOCAL_BUNDLE_ZIP.exists() else None,
        output_dir=REPO_ROOT / "data" / "fathomnet_underwater_tutorial_bundle",
    )

if "RUN_LIVE_TRAINING" not in globals():
    RUN_LIVE_TRAINING = bool(detect_runtime().get("has_cuda", False))

if "build_train_args" not in globals():
    def build_train_args(
        *,
        n_epochs=10,
        imgsz=320,
        batch=8,
        lr0=0.001,
        patience=3,
        workers=0,
        project="runs/tutorial",
        name="experiment",
        seed=42,
    ):
        '''Compact fallback copy of the visible helper from the setup section.'''

        return {
            "epochs": int(n_epochs),
            "imgsz": int(imgsz),
            "batch": int(batch),
            "lr0": float(lr0),
            "patience": int(patience),
            "workers": int(workers),
            "project": str(project),
            "name": str(name),
            "seed": int(seed),
            "save": True,
            "verbose": False,
        }


# Section bootstrap: segmentation
from scripts.tutorial_data import get_task_paths, select_yolo_examples, validate_yolo_dataset, yolo_label_instance_count
from scripts.tutorial_viz import draw_yolo_masks, plot_training_curves

SEGMENT_PATHS = get_task_paths("segment", BUNDLE_ROOT)
SEGMENT_ROOT = SEGMENT_PATHS["root"]
SEGMENT_YAML = SEGMENT_PATHS["yaml"]
CACHED_SEGMENT_RESULTS = BUNDLE_ROOT / "cached_training" / "segmentation" / "results.csv"

print(json.dumps(validate_yolo_dataset(SEGMENT_YAML, task="segment"), indent=2)[:3000])

### Segmentation Truth Example: Same Image, More Geometry

This plot shows **ground-truth polygon labels** from the YOLO segmentation dataset. When possible, it uses the same image as the detection example above so you can compare boxes with masks directly. The colored regions are annotation polygons, not model predictions.

In [ ]:
import matplotlib.pyplot as plt

preferred_stem = globals().get("DETECT_EXAMPLE_STEM")
if preferred_stem is not None:
    candidate_label = SEGMENT_ROOT / "labels" / "val" / f"{preferred_stem}.txt"
    candidate_image = SEGMENT_ROOT / "images" / "val" / f"{preferred_stem}.jpg"
else:
    candidate_label = None
    candidate_image = None

if candidate_label is not None and candidate_label.exists() and candidate_image.exists():
    first_segment_label = candidate_label
    first_segment_image = candidate_image
    segment_instance_count = yolo_label_instance_count(first_segment_label)
    print(f"Showing the same image as detection: {preferred_stem}")
else:
    segment_example = select_yolo_examples(SEGMENT_ROOT, split="val", min_instances=2, limit=1)[0]
    first_segment_label = segment_example["label_path"]
    first_segment_image = segment_example["image_path"]
    segment_instance_count = segment_example["instance_count"]
    print(f"Showing {first_segment_image.stem}: {segment_instance_count} truth masks")

ax = draw_yolo_masks(first_segment_image, first_segment_label, class_names={0: "object"})
ax.set_title(f"Segmentation truth labels: {segment_instance_count} masks")
plt.show()

### Visible Helper: Mask IoU

This helper is intentionally simple. It assumes binary masks with equal shape. You can modify it to handle soft masks, ignored pixels, or batches.

In [ ]:
def mask_iou(mask_a, mask_b):
    '''Compute IoU for two binary masks represented as nested lists or arrays.'''

    intersection = 0
    union = 0
    for row_a, row_b in zip(mask_a, mask_b):
        for value_a, value_b in zip(row_a, row_b):
            a = bool(value_a)
            b = bool(value_b)
            intersection += int(a and b)
            union += int(a or b)
    return intersection / union if union else 1.0


mask_a = [[1, 1, 0], [0, 1, 0], [0, 0, 0]]
mask_b = [[1, 0, 0], [0, 1, 1], [0, 0, 0]]
print("mask IoU:", mask_iou(mask_a, mask_b))

assert mask_iou([[0, 0]], [[0, 0]]) == 1.0
assert mask_iou([[1, 0]], [[0, 1]]) == 0.0
assert mask_iou([[1, 1]], [[1, 1]]) == 1.0

### Train Or Load A Small Segmentation Run

Now train the segmentation model, or load cached curves if live training is unavailable. The important difference from detection is that the report contains both box metrics and mask metrics. Compare them: a model can learn reasonable boxes before it learns precise masks. For a live run, the best validation checkpoint is saved as `weights/best.pt`.

In [ ]:
SEGMENT_N_EPOCHS = 10
SEGMENT_ARGS = build_train_args(
    n_epochs=SEGMENT_N_EPOCHS,
    imgsz=320,
    batch=4,
    lr0=0.001,
    project=REPO_ROOT / "runs" / "segment",
    name="segment_default",
)

# Documentation checkpoint:
# 1. Open https://docs.ultralytics.com/datasets/segment/
# 2. Confirm the polygon-row format and the dataset YAML structure.
# 3. Open https://docs.ultralytics.com/modes/train/ for the same model-loading pattern.
#
# Your live exercise is to connect the segmentation dataset YAML to a pretrained
# segmentation model and then interpret both box mAP and mask mAP.
segmentation_training = None
segmentation_save_dir = None
segmentation_best_model_path = None
if RUN_LIVE_TRAINING:
    from ultralytics import YOLO

    segmentation_model = YOLO("yolo11n-seg.pt")
    segmentation_training = segmentation_model.train(data=str(SEGMENT_YAML), **SEGMENT_ARGS)
    segmentation_save_dir = getattr(segmentation_training, "save_dir", None)
    if segmentation_save_dir is None and getattr(segmentation_model, "trainer", None) is not None:
        segmentation_save_dir = getattr(segmentation_model.trainer, "save_dir", None)
    print("Live segmentation training finished.")
    candidate_best = Path(segmentation_save_dir) / "weights" / "best.pt" if segmentation_save_dir is not None else None
    if candidate_best is not None and candidate_best.exists():
        segmentation_best_model_path = candidate_best
        print(f"Best validation checkpoint: {segmentation_best_model_path}")
else:
    print("No GPU detected. Using cached segmentation training curve.")

segmentation_results_csv = (
    Path(segmentation_save_dir) / "results.csv"
    if segmentation_save_dir is not None
    else CACHED_SEGMENT_RESULTS
)
if not segmentation_results_csv.exists():
    segmentation_results_csv = CACHED_SEGMENT_RESULTS

plot_training_curves(
    segmentation_results_csv,
    metric_columns=["metrics/mAP50(B)", "metrics/mAP50-95(B)", "metrics/mAP50(M)", "metrics/mAP50-95(M)"],
    title="Segmentation: box mAP vs mask mAP",
)

### Exercise: COCO Polygons To YOLO Segmentation Rows

COCO segmentation annotations often store polygon vertices as one flat list:

```text
[x1, y1, x2, y2, ..., xn, yn]
```

YOLO segmentation wants the same geometry normalized by image width and height, preceded by the class id.

You do not get the full converter here. The point is to reason through the coordinate map first, then decide what edge cases a production converter needs to handle: multiple polygons, holes, tiny regions, invalid polygons, and categories you might merge.

In [ ]:
def coco_polygon_to_yolo_row_exercise(class_id, polygon, image_width, image_height):
    '''Convert one COCO polygon list to one YOLO segmentation row.

    Return a list like `[class_id, x1, y1, x2, y2, ...]`.
    '''

    if len(polygon) < 6:
        raise ValueError("A YOLO segmentation polygon needs at least 3 points.")
    if len(polygon) % 2:
        raise ValueError("Polygon coordinates must be x/y pairs.")

    row = [int(class_id)]
    for index in range(0, len(polygon), 2):
        x = polygon[index] / image_width
        y = polygon[index + 1] / image_height
        row.extend([x, y])
    return row

example_polygon = [10, 10, 40, 10, 40, 30, 10, 30]
candidate_row = coco_polygon_to_yolo_row_exercise(0, example_polygon, image_width=100, image_height=100)

print("Your YOLO segmentation row:", candidate_row)
assert candidate_row == [0, 0.1, 0.1, 0.4, 0.1, 0.4, 0.3, 0.1, 0.3]
assert len(candidate_row) >= 7
assert len(candidate_row[1:]) % 2 == 0
assert all(0 <= value <= 1 for value in candidate_row[1:])

### Segmentation Exercises

Beginner:

- Inspect two masks: one clean and one messy.
- Why can box mAP be higher than mask mAP?
- Use the Ultralytics segmentation docs to identify the minimum number of polygon points per object.

Intermediate:

- Change the confidence threshold and inspect the visual result.
- Complete `mask_iou(...)` for one extra edge case: two empty masks, two disjoint masks, or two identical masks.
- Complete `coco_polygon_to_yolo_row_exercise(...)` above.

Advanced:

- Use the coarse biological label plan from the source YOLO repo as an after-session extension.
- Ask whether binary "object" segmentation is a scientifically useful target, or only a stepping stone.
- Propose a rule for dropping or keeping tiny polygons, then predict how that rule will affect recall and mAP.

### Master Notes: Segmentation

- YOLO segmentation requires at least three `(x, y)` polygon points per object.
- Box mAP can be higher than mask mAP because a predicted rectangle can be acceptable even when the boundary shape is poor.
- COCO polygon coordinates are normalized pairwise: `x/W`, `y/H`, preceded by the YOLO class id.
- Dropping tiny polygons often improves short workshop metrics by reducing ambiguous labels, but it can reduce ecological recall for small organisms.
- Binary object segmentation is useful as a stepping stone and for generic saliency, but it is less scientifically expressive than taxonomic or functional-group labels.

# Part 4: SAM3 Text-Prompt Segmentation Lab

SAM3 changes the interface. Instead of training a new model for every label set, you can ask for a concept:

```text
image + "fish" -> masks, boxes, scores
```

Mathematically, the text prompt becomes part of the input:

$$g_\phi(x,q)\rightarrow \{(M_i,b_i,s_i)\}_{i=1}^m.$$

That changes the main question from "how do you train a new supervised head?" to "which prompt and threshold define the object concept you actually want?"

The default path below uses cached SAM3-like outputs so everyone can complete the lab. If your runtime passes the live SAM3 checks and you have checkpoint access, set `USE_LIVE_SAM3 = True`.

### Section Bootstrap: Load SAM3 Cache And Runtime Checks

This cell checks whether live SAM3 is possible and lists the cached prompt outputs available in the tutorial bundle. The rest of the section works even when live SAM3 is unavailable.

In [ ]:
# Common skip-ahead bootstrap. This block lets a section run even if you did
# not execute the setup cells above.
from pathlib import Path
import json
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/Spiffical/fathomnet-underwater-vision-tutorial"
GITHUB_BRANCH = "main"
PROJECT_DIR_NAME = "fathomnet_underwater_vision_tutorial"
BUNDLE_URL = "https://github.com/Spiffical/fathomnet-underwater-vision-tutorial/raw/main/data/fathomnet_underwater_tutorial_bundle.zip"

if "REPO_ROOT" not in globals():
    def _candidate_roots():
        cwd = Path.cwd().resolve()
        yield cwd
        yield from cwd.parents
        yield Path("/content") / PROJECT_DIR_NAME

    REPO_ROOT = None
    for candidate in _candidate_roots():
        if (candidate / "scripts").exists() and (candidate / "notebooks").exists():
            REPO_ROOT = candidate
            break
    if REPO_ROOT is None and "google.colab" in sys.modules:
        clone_dir = Path("/content") / PROJECT_DIR_NAME
        if not clone_dir.exists():
            subprocess.check_call(["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, f"{GITHUB_REPO_URL}.git", str(clone_dir)])
        REPO_ROOT = clone_dir
    if REPO_ROOT is None:
        raise RuntimeError("Could not find the tutorial repo root. Start Jupyter from the cloned repo.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.tutorial_setup import detect_runtime, download_tutorial_bundle, ensure_dependencies

if "LOCAL_BUNDLE_ZIP" not in globals():
    LOCAL_BUNDLE_ZIP = REPO_ROOT / "data" / "fathomnet_underwater_tutorial_bundle.zip"

ensure_dependencies(install=("google.colab" in sys.modules), extra_pip_args=("--quiet",))

if "BUNDLE_ROOT" not in globals():
    BUNDLE_ROOT = download_tutorial_bundle(
        bundle_url=BUNDLE_URL or None,
        bundle_zip_path=LOCAL_BUNDLE_ZIP if LOCAL_BUNDLE_ZIP.exists() else None,
        output_dir=REPO_ROOT / "data" / "fathomnet_underwater_tutorial_bundle",
    )

if "RUN_LIVE_TRAINING" not in globals():
    RUN_LIVE_TRAINING = bool(detect_runtime().get("has_cuda", False))

if "build_train_args" not in globals():
    def build_train_args(
        *,
        n_epochs=10,
        imgsz=320,
        batch=8,
        lr0=0.001,
        patience=3,
        workers=0,
        project="runs/tutorial",
        name="experiment",
        seed=42,
    ):
        '''Compact fallback copy of the visible helper from the setup section.'''

        return {
            "epochs": int(n_epochs),
            "imgsz": int(imgsz),
            "batch": int(batch),
            "lr0": float(lr0),
            "patience": int(patience),
            "workers": int(workers),
            "project": str(project),
            "name": str(name),
            "seed": int(seed),
            "save": True,
            "verbose": False,
        }


# Section bootstrap: SAM3
from scripts.tutorial_sam3 import (
    available_cached_prompts,
    load_cached_sam3_result,
    run_sam3_text_prompt,
    sam3_can_run_live,
)
from scripts.tutorial_viz import plot_sam3_result

SAM3_STATUS = sam3_can_run_live()
USE_LIVE_SAM3 = False

print(json.dumps(SAM3_STATUS, indent=2))
print(json.dumps(available_cached_prompts(BUNDLE_ROOT), indent=2)[:3000])

### Visible Helper: Try A SAM3 Prompt

Edit `prompt` and `confidence_threshold`. The same function works with cached fallback outputs or live SAM3.

In [ ]:
def try_sam3_prompt(image_id, prompt, confidence_threshold=0.5):
    '''Run or load SAM3-style text-prompt segmentation for one image.'''

    cached = load_cached_sam3_result(BUNDLE_ROOT, image_id=image_id, prompt=prompt)
    image_path = cached["image_path"]

    if USE_LIVE_SAM3:
        live = run_sam3_text_prompt(
            image_path,
            prompt,
            confidence_threshold=confidence_threshold,
        )
        live["image_path"] = image_path
        return live

    cached["confidence_threshold"] = confidence_threshold
    return cached


cached_prompt_table = available_cached_prompts(BUNDLE_ROOT)
SAM3_IMAGE_ID = next(iter(cached_prompt_table))
PROMPT = "marine organism"
CONFIDENCE_THRESHOLD = 0.5

sam3_result = try_sam3_prompt(SAM3_IMAGE_ID, PROMPT, CONFIDENCE_THRESHOLD)
plot_sam3_result(
    sam3_result["image_path"],
    sam3_result,
    score_threshold=CONFIDENCE_THRESHOLD,
    title=f"{SAM3_IMAGE_ID}: {PROMPT}",
)

### Compare Several Prompts

After one prompt works, try several prompts on the same cached image. This cell does not train anything; it only changes the text query and reports how many masks were returned for each concept.

In [ ]:
# Try several prompts on the same image. Some may correctly return nothing.
for prompt in ["fish", "sponge", "gelatinous animal", "small crab", "echinoderm"]:
    result = try_sam3_prompt(SAM3_IMAGE_ID, prompt, confidence_threshold=0.5)
    print(prompt, "detections:", len(result.get("boxes", [])), "source:", result.get("source"))

### SAM3 Exercises

Beginner:

- Try `PROMPT = "fish"`, `"sponge"`, `"gelatinous animal"`, and `"small crab"`.
- Increase `CONFIDENCE_THRESHOLD` from `0.5` to `0.8`.
- Which prompts are too broad? Which are too specific?

Advanced:

- If live SAM3 works, compare cached fallback with live masks.
- Try using SAM3 outputs as pseudo-labels, then reason about when pseudo-label noise helps or hurts supervised training.

### Master Notes: SAM3

- Broad prompts such as `marine organism` tend to increase recall and false positives.
- Specific prompts such as `small crab` can improve precision when the concept is visually present, but often return nothing when the image lacks the target or the prompt misses the model's visual vocabulary.
- Raising `CONFIDENCE_THRESHOLD` filters low-score masks; expect fewer detections, higher apparent precision, and lower recall.
- SAM3 pseudo-labels are useful when they reduce annotation effort, but noisy pseudo-labels can teach a supervised model the prompt model's mistakes.

# Wrap-Up

You moved through four levels of supervision and geometry:

- classification: one image crop, one class;
- detection: objects as normalized boxes;
- segmentation: objects as masks or polygons;
- SAM3 prompting: segmentation conditioned on text.

Good after-session projects:

- train on the 500-image or full dataset after this notebook,
- convert the coarse biological label plan into a multiclass detection task,
- compare YOLO training against FathomNet pretrained checkpoints,
- use SAM3 prompt outputs to propose labels for active learning,
- extend from images to video tracking.

The main lesson is not that one model wins everywhere. The real modeling choice is the pair:

$$\text{scientific question} \quad + \quad \text{label geometry}.$$

Once that pair is clear, the model choice becomes much easier to reason about.